# MORQEX geometry workflow / MORQEX几何建模

256³ original voxels, live2D/3D and distinct voxel-reference, raw-TetGen and GEMsim-adaptive workflows. Execute a copy outside source.
原始256³体素、实时二维/三维及独立的体素参考、原始TetGen、GEMsim自适应网格；请在源码外执行副本。

In [ ]:
from pathlib import Path
import sys,os
SOURCE=Path('/home/zephyrain/lw/MORQEX')
sys.path.insert(0,str(SOURCE/'src'))
WORK=Path('/home/zephyrain/lw/morph_workspace/microelectrode_gallery/notebook_example')
WORK.mkdir(parents=True,exist_ok=True)
import numpy as np
from morqex.modeling import from_regions,Shape,from_shapes


## Live view / 实时预览
The same display handle is updated during construction.
生成过程复用同一显示句柄。

In [ ]:
blank=from_regions(np.zeros((256,)*3,np.uint8),spacing=(0.5e-6,)*3)
view=blank.show(maximum_3d_size=40,height=680,title='Live geometry / 实时几何')


In [ ]:
rng=np.random.default_rng(20261127)
objects=[]
for k in range(18):
    angle=float(rng.uniform(-0.8,0.8))
    R=np.array([[np.cos(angle),0,np.sin(angle)],[0,1,0],[-np.sin(angle),0,np.cos(angle)]])
    objects.append(Shape('capsule',1,tuple(rng.uniform(15e-6,113e-6,3)),(4e-6,4e-6,float(rng.uniform(12e-6,30e-6))),tuple(map(tuple,R)),0.2))
model=from_shapes(objects,shape=(256,)*3,spacing=(0.5e-6,)*3,progress=view.update,progress_every=6)
view.update(model)
view.save_html(WORK/'live_geometry.html')


## Full tetrahedral mesh / 完整体四面体网格
The full256³ array is meshed; displayLOD does not enter the mesh calculation.
对完整256³数组生成网格，预览LOD不参与计算。

In [ ]:
from datetime import datetime, timezone
# Each mesh call snapshots its source and uses a fresh output directory.
# 每次网格化保存独立源体素，避免参数改变后误用旧网格。
MESH_DIR=WORK/"voxel_mesh"/datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
MESH_DIR.mkdir(parents=True)
target=MESH_DIR/"mesh.h5"
model.save(MESH_DIR/"source_voxels.h5")
artifact=model.mesh_to(target,regions=[1],slab_depth=2)
print(artifact)


In [ ]:
print("HTML:",WORK/"live_geometry.html")
print("Mesh source / 网格对应源体素:",MESH_DIR/"source_voxels.h5")
print("ParaView:",target.with_suffix(".xdmf"))


In [ ]:
from morqex.modeling import MeshArtifact
saved=MeshArtifact.load(target)
mesh_view=saved.show(first_cell=0,cell_count=600,height=620)
mesh_view.save_html(MESH_DIR/"actual_mesh.html")


## Native TetGen + COMSOL / 原生 TetGen 与 COMSOL

This is a separate24³ multiphase example, **not** a replacement for the preceding full256³ voxel mesh.
此处是独立24³多相示例，不冒充前面完整256³体素网格。
Compile once using `MORQEX_BUILD_TETGEN=1 pip install .`, or the explicit external build shown below.
先用可选 pip 编译，或显式执行下面的外部构建；不在源码内存放二进制。
See [the bilingual tutorial](../05_tetgen_and_comsol.md) for all mesh routes and quality limits.


In [ ]:
from morqex.modeling import (build_tetgen,from_toml,mesh_from_toml,export_comsol,read_comsol)
from morqex.modeling.tetgen_backend import library_path
from datetime import datetime,timezone
os.environ.setdefault('MORQEX_WORKSPACE',str(WORK.parent))
try:
    print('Native library:',library_path())
except RuntimeError:
    print('Explicit one-job external build:',build_tetgen(WORK.parent))
recipe=SOURCE/'configs/geometry_tetgen.toml'
native_model=from_toml(recipe)
native_mesh=mesh_from_toml(native_model,recipe)
print(native_mesh.report)
assert native_mesh.signed_volumes().min()>0


In [ ]:
from IPython.display import display
native_dir=WORK/'tetgen'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
native_dir.mkdir(parents=True)
display(native_model.viewer(maximum_3d_size=32,title='24³ source regions / 原始分区'))
display(native_mesh.viewer(cell_count=2000,title='Actual native TetGen cells / 真实 TetGen 单元'))
native_model.viewer(maximum_3d_size=32).save_html(native_dir/'voxels.html')
native_mesh.viewer(cell_count=2000).save_html(native_dir/'mesh.html')
native_artifact=native_mesh.save_hdf5(native_dir/'mesh.h5')
comsol_file=export_comsol(native_mesh,native_dir/'mesh.mphtxt',region_names=native_model.region_names)
back=read_comsol(comsol_file)
np.testing.assert_array_equal(back['points'],native_mesh.points)
np.testing.assert_array_equal(back['elements']['tet']['cells'],native_mesh.cells)
print('COMSOL format/numerical read-back passed / 格式与数值读回通过:',comsol_file)
print('Actual COMSOL GUI import was not run / 未执行实际 COMSOL GUI 导入')


## GEMsim weighted ACVD + true volume sections / 加权自适应曲面与真实体切面

This separate full256³ example uses the new resolved-feature recipe and original weighted ACVD kernel; the TOML file supplies meshing options only.
此处是独立完整256³可执行示例，通过API明确指定几何；TOML只提供网格选项。完整十种256³案例由批量图库脚本生成。
The earlier voxel-split and raw-TetGen examples are retained as different methods.
前面的体素剖分及原始TetGen示例保留，方法之间不混淆。


In [ ]:
from morqex.modeling import build_acvd,AdaptiveMeshOptions,resolved_electrode
from morqex.modeling.acvd_backend import load_acvd
try:
    print('Weighted ACVD:',load_acvd().__file__)
except RuntimeError:
    # Explicit build requires Cython / 显式源码编译需要Cython
    print('Explicit single-job build:',build_acvd(WORK.parent))
adaptive_recipe=SOURCE/'configs/geometry_adaptive.toml'
adaptive_model=resolved_electrode('granular',size=256,spacing=0.5e-6,seed=20261127,cbd_thickness_voxels=4,minimum_feature_voxels=4)
adaptive_result=mesh_from_toml(adaptive_model,adaptive_recipe)
print(adaptive_result.report)
assert set(adaptive_result.phases)=={0,1,2}
assert adaptive_result.signed_volumes().min()>0


In [ ]:
adaptive_dir=WORK/'adaptive'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
adaptive_dir.mkdir(parents=True)
display(adaptive_model.viewer(maximum_3d_size=40,title='Source active/CBD/pore / 原始三相结构'))
display(adaptive_result.adaptive_viewer(title='Weighted ACVD surface + real volume sections / 自适应曲面与真实体切面'))
adaptive_model.save(adaptive_dir/'geometry.h5')
adaptive_result.save_hdf5(adaptive_dir/'mesh.h5')
adaptive_result.adaptive_viewer().save_html(adaptive_dir/'adaptive.html')
export_comsol(adaptive_result,adaptive_dir/'mesh.mphtxt',region_names=adaptive_model.region_names)
print('Actual adaptive artifacts / 自适应结果:',adaptive_dir)
print('Quality requests met:',adaptive_result.report['quality_limits_satisfied'])


### Anisotropic voxels and matching physical sections / 非等距体素与同位置切面

Pixel dimensions follow SI spacing. `lower` uses Cartesian vertical orientation. The XYZ fractions below pass through the default voxel centres; choose XY in the mesh viewer to compare the Z slice. This is a geometric example, not calibrated material data.

二维像素按实际边长绘制，`lower`对应笛卡尔方向。下面的XYZ相对坐标穿过默认体素中心；在网格查看器选择XY切面可与Z切片对照。这是几何演示，不是材料标定数据。


In [ ]:
from morqex.modeling import from_regions,tetgen_mesh,TetGenOptions
anisotropic_labels=np.zeros((12,16,20),dtype=np.uint8)
anisotropic_labels[2:10,3:13,4:16]=1
anisotropic_model=from_regions(anisotropic_labels,spacing=(0.5e-6,1e-6,2e-6),
    origin=(5e-6,-2e-6,3e-6),region_names={0:'matrix',1:'inclusion'})
display(anisotropic_model.viewer(maximum_3d_size=24,slice_origin='lower',
    title='Physical voxel aspect / 真实体素比例'))
anisotropic_mesh=tetgen_mesh(anisotropic_model,
    options=TetGenOptions(maximum_cells=100000,isolated=True))
fractions_xyz=tuple((n//2+.5)/n for n in anisotropic_model.regions.shape[::-1])
display(anisotropic_mesh.adaptive_viewer(section_fractions=fractions_xyz,
    title='Matching voxel-centre planes / 同一体素中心切面'))
print('XYZ section fractions:',fractions_xyz)


## Complete CT and configurable software units / 完整CT与软件单位

The TIFF and COMSOL references are independent. Preserve source labels. Internal geometry is SI; exported values use the chosen unit. This cell previews full source arrays with explicit display LOD; it does not run the expensive volume-mesh qualification.
TIFF与COMSOL参考互不对应。原始分区保持不变，内部用米；导出单位只换算数值。下列预览保留完整源数组，显示LOD明确，不会自动启动大规模体网格求解。


In [ ]:
from morqex.modeling import tiff_voxels, VoxelModel, coated_electrode
ct_path=SOURCE.parent/'NMC.tif'
if ct_path.exists():
    original_ct=tiff_voxels(ct_path,spacing=(0.5e-6,)*3)
    print('Full source ZYX:',original_ct.regions.shape,original_ct.region_names)
    original_ct.viewer(maximum_3d_size=48,title='Original CT gray labels / 原始CT分区').display()
else:
    print('Optional local CT absent; synthetic modelling remains available / 未找到可选CT')
coated_path=WORK.parent/'nmc_complex_gallery/native/rods/geometry.h5'
coated=VoxelModel.load(coated_path) if coated_path.exists() else coated_electrode('rods',size=256,seed=20261127)
coated.viewer(maximum_3d_size=48,title='Native full256³ irregular rods / 原生不规则长条').display()


In [ ]:
# A small independent unit test, not a surrogate for the full CT volume.
# 独立小型单位检查，不冒充完整CT有限元网格。
unit_model=from_regions(np.ones((3,4,5),np.uint8),spacing=(0.2,)*3)
unit_mesh=unit_model.tetrahedralize()
unit_folder=WORK/'unit_checks'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
unit_folder.mkdir(parents=True)
for name,scale in [('m',1.),('mm',1e-3),('um',1e-6)]:
    target=export_comsol(unit_mesh,unit_folder/f'mesh_{name}.mphtxt',length_unit=name,
        selections={'whole_electrode':{'regions':[1]}})
    back=read_comsol(target)
    np.testing.assert_allclose(back['points']*scale,unit_mesh.points)
    assert back['selections']['whole_electrode']['entities']==[1]
    print(name,'numeric XYZ extent:',np.ptp(back['points'],axis=0))


## Fixed-grid smoothing is not resolution reduction / 原尺寸平滑不是降低分辨率

This independent32³ example checks both operations and displays actual changes. It is not a substitute for a full256³ CT mesh. For full-size code, controls, licenses and qualification limits see tutorial08.
独立32³例子检查两种操作，并显示真实变化；不冒充完整256³ CT网格。完整尺寸示例、选项与限制见教程08。


In [ ]:
from morqex.modeling import (smooth_voxels, SmoothingOptions, lower_resolution,
    NotebookProgress, recommend_mesh_sizes, ImplicitMeshOptions, implicit_mesh)
z,y,x=np.indices((32,32,32))
labels=np.zeros(z.shape,np.uint8)
labels[(z-15.5)**2+(y-15.5)**2+(x-15.5)**2<10**2]=128
labels[13:19,13:19,13:19]=255
labels[4:7,15,15]=128
raw32=from_regions(labels,spacing=(0.5e-6,)*3)
progress=NotebookProgress()
p32=smooth_voxels(raw32,options=SmoothingOptions(sigma_voxels=.7,continuous_volume_calibration=True),progress=progress)
assert p32.model.regions.shape==raw32.regions.shape
assert p32.model.spacing==raw32.spacing
print('Fixed-grid / 原尺寸:',p32.report['source_shape_zyx'],p32.report['processed_shape_zyx'])
print('Changed fraction / 改变比例:',p32.report['fidelity']['changed_fraction'])
display(raw32.viewer(maximum_3d_size=32,title='Original32³ / 原始32³'))
display(p32.model.viewer(maximum_3d_size=32,title='Smoothed32³, same physical box / 平滑32³，范围不变'))


In [ ]:
reduced32,rr=lower_resolution(raw32,factor=2,progress=progress)
assert reduced32.regions.shape==(16,16,16)
np.testing.assert_allclose(np.array(raw32.regions.shape)*raw32.spacing,
                           np.array(reduced32.regions.shape)*reduced32.spacing)
print('Lower resolution / 降低分辨率:',rr)
display(reduced32.viewer(maximum_3d_size=16,title='16³ reduced sampling, same extent / 16³降采样，范围不变'))
print('Mesh advice / 网格尺寸建议:',recommend_mesh_sizes(p32,progress=progress)['recommended_sizes'])


In [ ]:
# The small nested phase may need a second bounded geometry refinement.
# 小体积分数嵌套相允许额外一次有限几何加密，不改变相体积容差。
# Compile explicitly before this cell: build_cgal(WORK.parent, dependency_prefix=...).
# 运行前明确编译 CGAL，不在导入或绘图时自动下载/编译。
from morqex.modeling.cgal_backend import library_path as cgal_library
print('CGAL:',cgal_library(WORK.parent))
implicit_dir=WORK/'implicit'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
implicit_dir.mkdir(parents=True)
p32.save(implicit_dir/'prepared.h5')
counts=[]
for precision in ('coarse','fine'):
    im=implicit_mesh(p32,workspace=WORK.parent,
        options=ImplicitMeshOptions(precision=precision,isolated=True,maximum_quality_retries=2),progress=progress)
    assert im.report['geometry_score_sha256']==p32.report['score_sha256']
    assert im.report['quality_limits_satisfied']
    counts.append(len(im.cells))
    ia=im.save_hdf5(implicit_dir/(precision+'.h5'))
    display(ia.adaptive_viewer(maximum_embedded_faces=100000))
    ia.adaptive_viewer(maximum_embedded_faces=100000).save_html(implicit_dir/(precision+'.html'))
assert counts[1]>counts[0]
print('Same prepared32³, coarse/fine cells / 同一准备几何粗细网格:',counts)
